In [5]:
# scripts/make_map.py
"""
Interactive world map (no Mapbox) + accessible page

Inputs (repo-relative):
- data/processed/data_clean.csv
- data/geo/countries.geojson.json  (or countries.geojson)
- (optional) data/itc_countries.txt  # one country per line, for denominator in ITC stats

Join key:
- GeoJSON country *name* (not ISO). Matching: exact (CI) → alias → fuzzy.

Outputs:
- docs/interactive_country_map.html  (interactive Plotly map)
- docs/map_accessible.html           (summary + full tables)
- outputs/name_match_review.csv      (how each name was matched)
- outputs/unmapped_entities.csv      (names not matched → education/fixes)

Design:
- Two choropleth layers: zeros (grey, bottom) + non-zeros (blue, top)
- ITC is a metric (1/0). Selecting “ITC countries” in the dropdown
  highlights only ITC countries; other metrics don’t show ITC highlighting,
  but the hover always ends with “ITC country: Yes/No”.
"""

from __future__ import annotations
from pathlib import Path
import json, re, sys
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "browser"  # opens your browser locally


# ---------- repo root detection ----------
def _find_repo_root() -> Path:
    try:
        here = Path(__file__).resolve()
        return here.parent.parent   # …/scripts → repo root
    except NameError:               # notebooks / REPL
        cwd = Path.cwd().resolve()
        if (cwd / "data").is_dir() and (cwd / "scripts").is_dir():
            return cwd
        if cwd.name == "scripts" and (cwd.parent / "data").is_dir():
            return cwd.parent
        cur = cwd
        for _ in range(5):
            if (cur / ".git").is_dir() or ((cur / "data").is_dir() and (cur / "scripts").is_dir()):
                return cur
            cur = cur.parent
        return cwd


ROOT = _find_repo_root()
CSV_PATH = ROOT / "data" / "processed" / "data_clean.csv"
GEO_DIR  = ROOT / "data" / "geo"
GJ_PATH  = GEO_DIR / ("countries.geojson.json" if (GEO_DIR / "countries.geojson.json").exists()
                      else "countries.geojson")
ITC_TXT_PATH = ROOT / "data" / "itc_countries.txt"  # optional; for denominator in summary

DOCS_DIR = ROOT / "docs"
OUT_DIR  = ROOT / "outputs"
DOCS_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)


# ---------- helpers ----------
def yn_to_bool(s: pd.Series) -> pd.Series:
    return (
        s.astype(str).str.strip().str.lower()
         .map({"y": True, "yes": True, "member": True, "1": True, "true": True, "t": True, "x": True,
               "n": False, "no": False, "0": False, "false": False, "": False})
         .fillna(False)
    )

def normalize_name(s: str) -> str:
    s = (str(s) or "").strip()
    s = re.sub(r"\s*\([^)]*\)\s*$", "", s)
    s = s.replace("’", "'")
    s = " ".join(s.split())
    return s.lower()

def load_itc_list(path: Path) -> set[str] | None:
    if not path.exists():
        return None
    items = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#"):
            items.append(line)
    return set(items) if items else None


# ---------- load data ----------
if not CSV_PATH.exists():
    sys.exit(f"ERROR: {CSV_PATH.relative_to(ROOT)} not found. Run scripts/make_eda.py first.")

with open(GJ_PATH, "r", encoding="utf-8") as f:
    gj = json.load(f)

print("Repo root →", ROOT)
print("Using CSV  →", CSV_PATH.relative_to(ROOT))
print("Using GEO  →", Path(GJ_PATH).relative_to(ROOT))

df = pd.read_csv(CSV_PATH, low_memory=False)

# choose the best name-like property in the GeoJSON
prop0 = gj["features"][0]["properties"]
name_key_candidates = ["name", "NAME", "ADMIN", "NAME_EN", "COUNTRY"]
name_key = next((k for k in name_key_candidates if k in prop0), None)
if not name_key:
    raise ValueError("Couldn't find a name-like property in GeoJSON. Tried: " + ", ".join(name_key_candidates))

GEO_NAMES = {feat["properties"][name_key] for feat in gj["features"]}
GEO_NAMES_LIST = sorted(GEO_NAMES)
GEO_NAME_MAP_LC = {n.lower(): n for n in GEO_NAMES}  # case-insensitive exact lookup


# ---------- detect columns & gentle clean ----------
country_col = "country_clean" if "country_clean" in df.columns else ("country" if "country" in df.columns else None)
if not country_col:
    raise ValueError("No 'country_clean' or 'country' column found in data_clean.csv.")

mc_col   = "mc_member"  if "mc_member"  in df.columns else None
core_col = "core_group" if "core_group" in df.columns else None

wg_cols = [c for c in df.columns if re.match(r"(?i)^wg\d", c)]
wg_cols = sorted(wg_cols, key=lambda x: int(re.findall(r"\d+", x)[0]) if re.findall(r"\d+", x) else 99)

wg_member_col = "wg_member" if "wg_member" in df.columns else None

# normalize NBSP + trim in object columns
for c in df.select_dtypes(include="object").columns:
    df[c] = df[c].astype(str).str.replace("\u00A0", " ", regex=False).str.strip()

# coerce flags
if mc_col:
    if df[mc_col].dtype != bool:
        df[mc_col] = yn_to_bool(df[mc_col])
else:
    df["mc_member"] = False
    mc_col = "mc_member"

if core_col:
    if df[core_col].dtype != bool:
        df[core_col] = yn_to_bool(df[core_col])
else:
    df["core_group"] = False
    core_col = "core_group"

for c in wg_cols:
    if df[c].dtype != bool:
        df[c] = yn_to_bool(df[c])

df["any_wg"] = df[wg_cols].any(axis=1) if wg_cols else False
if wg_member_col:
    df["any_wg"] = df["any_wg"] | yn_to_bool(df[wg_member_col])

# Normalize itc_countries (if present) to Yes/No (string) for clarity
have_itc = "itc_countries" in df.columns
if have_itc:
    itc_norm = df["itc_countries"].astype(str).str.strip().str.lower()
    df["itc_countries"] = np.where(itc_norm.isin(["yes", "y", "true", "1"]), "Yes", "No")
    ITC_LIST = load_itc_list(ITC_TXT_PATH)  # optional, for denominator in summary
else:
    ITC_LIST = None


# ---------- name matching: exact (CI) → alias → fuzzy ----------
try:
    from rapidfuzz import process as rf_process, fuzz as rf_fuzz
    HAVE_RF = True
except Exception:
    from difflib import get_close_matches, SequenceMatcher
    HAVE_RF = False

ALIASES_TO_GJ = {
    "kosovo*": "Kosovo", "kosovo": "Kosovo", "republic of kosovo": "Kosovo",
    "czech republic": "Czechia",
    "macedonia": "North Macedonia",
    "serbia": "Republic of Serbia",
    "united states": "United States of America",
    "usa": "United States of America",
    "uk": "United Kingdom",
    "türkiye": "Turkey", "turkiye": "Turkey", "turkey": "Turkey",
    "côte d’ivoire": "Ivory Coast", "cote d'ivoire": "Ivory Coast",
}
STOPLIST = {s.lower() for s in {
    "European Commission and EU Agencies",
    "European RTD Organisations",
    "European Commission",
    "European Union", "EU",
}}

def fuzzy_best(name_clean_lc: str):
    if HAVE_RF:
        match = rf_process.extractOne(name_clean_lc, GEO_NAMES_LIST, scorer=rf_fuzz.WRatio)
        if match is None:
            return None, 0.0
        cand, score, _ = match
        return cand, score / 100.0
    cand = get_close_matches(name_clean_lc, GEO_NAMES_LIST, n=1, cutoff=0.0)
    if not cand:
        return None, 0.0
    score = SequenceMatcher(None, name_clean_lc, cand[0]).ratio()
    return cand[0], score

FUZZY_THRESHOLD = 0.85
_cache = {}

def to_geojson_name(raw: str):
    if pd.isna(raw):
        return None
    if raw in _cache:
        return _cache[raw]
    nrm = normalize_name(raw)
    if nrm in STOPLIST:
        _cache[raw] = None
        return None
    exact = GEO_NAME_MAP_LC.get(nrm)
    if exact:
        _cache[raw] = exact
        return exact
    alias = ALIASES_TO_GJ.get(nrm)
    if alias:
        _cache[raw] = alias
        return alias
    cand, score = fuzzy_best(nrm)
    if cand and score >= FUZZY_THRESHOLD:
        _cache[raw] = cand
        return cand
    _cache[raw] = None
    return None

df["gj_name"] = df[country_col].apply(to_geojson_name)

# review how names were matched (education / QA)
unique_raw = (df[country_col].dropna().map(str).map(str.strip).drop_duplicates().sort_values())
rows = []
for raw in unique_raw:
    nrm = normalize_name(raw)
    exact_ci = GEO_NAME_MAP_LC.get(nrm)
    alias    = ALIASES_TO_GJ.get(nrm)
    cand, score = fuzzy_best(nrm)
    chosen   = to_geojson_name(raw)
    method   = ("exact" if exact_ci and chosen == exact_ci else
                "alias" if alias and chosen == alias else
                "fuzzy" if cand and chosen == cand else
                "none")
    rows.append({
        "raw": raw,
        "normalized_lower": nrm,
        "exact_match_ci": exact_ci,
        "alias_used": alias,
        "fuzzy_candidate": cand,
        "fuzzy_score": round(score, 3),
        "chosen": chosen,
        "method": method
    })
pd.DataFrame(rows).to_csv(OUT_DIR / "name_match_review.csv", index=False)
print("Saved →", (OUT_DIR / "name_match_review.csv").relative_to(ROOT))


# ---------- aggregate to country ----------
agg_parts = {
    "gj_name":      ("gj_name", "first"),
    "total_people": (country_col, "size"),
    "MC":           (mc_col, "sum"),
    "Core":         (core_col, "sum"),
    "Any_WG":       ("any_wg", "sum"),
}
for c in wg_cols:
    agg_parts[c] = (c, "sum")

agg = df.groupby("gj_name", dropna=False, as_index=False).agg(**agg_parts)
count_cols = ["total_people", "MC", "Core", "Any_WG"] + wg_cols
for c in count_cols:
    agg[c] = agg[c].fillna(0).astype(int)

not_in_geojson = (agg.loc[agg["gj_name"].isna() | ~agg["gj_name"].isin(GEO_NAMES), "gj_name"]
                    .dropna().sort_values().unique().tolist())
if not_in_geojson:
    pd.DataFrame({"not_in_geojson": not_in_geojson}).to_csv(OUT_DIR / "unmapped_entities.csv", index=False)
    print("Saved →", (OUT_DIR / "unmapped_entities.csv").relative_to(ROOT))

agg_map = agg[agg["gj_name"].isin(GEO_NAMES)].copy()

# ---------- ITC per-country (1/0) + label ----------
if have_itc:
    itc_by_country = (
        df.dropna(subset=["gj_name"])
          .groupby("gj_name")["itc_countries"]
          .apply(lambda s: s.astype(str).str.strip().str.lower().isin(["yes","y","true","1"]).any())
    )
    agg_map["ITC_flag"]  = agg_map["gj_name"].map(lambda n: 1 if bool(itc_by_country.get(n, False)) else 0).astype(int)
    agg_map["ITC_label"] = agg_map["ITC_flag"].map({1: "Yes", 0: "No"})
else:
    agg_map["ITC_flag"]  = 0
    agg_map["ITC_label"] = "No"

# ---------- plot state ----------
map_labels = {"total_people": "Total people",
              "MC": "Management Committee",
              "Core": "Core Group",
              "Any_WG": "Any Working Group"}
for c in wg_cols:
    m = re.search(r"\d+", c)
    n = m.group(0) if m else c
    map_labels[c] = f"Working Group (WG) {n}"

# Add ITC metric/label
map_labels["ITC_flag"] = "ITC countries"

table_labels = {"gj_name": "Country", "total_people": "Total", "MC": "MC", "Core": "Core", "Any_WG": "Any WG"}
for c in wg_cols:
    m = re.search(r"\d+", c)
    n = m.group(0) if m else c
    table_labels[c] = f"wg{n}"
table_labels["ITC_flag"] = "ITC"

metrics = list(map_labels.keys())
metric_totals = {m: int(agg_map[m].sum()) for m in metrics if m in agg_map.columns}

# Hover customdata: include ITC_label at the end
custom_cols = ["gj_name"] + count_cols + ["ITC_label"]
ITC_IDX = len(custom_cols) - 1

hover_lines = [
    "<b>%{customdata[0]}</b>",
    "Total: %{customdata[1]}",
    "MC: %{customdata[2]}",
    "Core: %{customdata[3]}",
    "Any WG: %{customdata[4]}",
]
# add WG lines (positions start at 5)
for i, c in enumerate(wg_cols, start=5):
    m = re.search(r"\d+", c)
    n = m.group(0) if m else c
    hover_lines.append(f"Working Group (WG) {n}: %{{customdata[{i}]}}")

# ITC Yes/No at the end
hover_lines.append(f"ITC country: %{{customdata[{ITC_IDX}]}}")
hovertemplate = "<br>".join(hover_lines) + "<extra></extra>"

def build_state(metric: str):
    vals = agg_map.set_index("gj_name")[metric]
    nz = vals[vals > 0]
    z  = vals[vals <= 0]
    base = agg_map.set_index("gj_name", drop=False)
    cd_cols = custom_cols
    return dict(
        loc_nz = nz.index.tolist(),
        z_nz   = nz.values,
        cd_nz  = base.loc[nz.index, cd_cols].values,
        loc_z  = z.index.tolist(),
        z_z    = [0] * len(z),
        cd_z   = base.loc[z.index, cd_cols].values,
    )

STATE = {m: build_state(m) for m in metrics}

# Color scale cap to 98th percentile
PCT_CAP = 98
caps = {}
for m in metrics:
    z = np.asarray(STATE[m]["z_nz"], dtype=float)
    cap = np.percentile(z, PCT_CAP) if z.size else 1.0
    if not np.isfinite(cap) or cap <= 0:
        cap = max(float(z.max()) if z.size else 1.0, 1.0)
    caps[m] = float(cap)


# ---------- figure ----------
LAND_GRAY  = "#E6E6E6"
HEAT_SCALE = [(0.00,"#9CC0FF"), (0.30,"#6FA5FF"), (0.60,"#3F7EE6"), (0.85,"#1F5FCC"), (1.00,"#0B42C1")]

initial_metric = "Any_WG"
s = STATE[initial_metric]
fig = go.Figure()

# base: zero-value countries
fig.add_trace(go.Choropleth(
    geojson=gj, featureidkey=f"properties.{name_key}",
    locations=s["loc_z"], z=s["z_z"], text=s["loc_z"], customdata=s["cd_z"],
    hovertemplate=hovertemplate, showscale=False,
    colorscale=[(0, LAND_GRAY), (1, LAND_GRAY)],
    marker_line_color="white", marker_line_width=0.5
))

# top: non-zero countries
fig.add_trace(go.Choropleth(
    geojson=gj, featureidkey=f"properties.{name_key}",
    locations=s["loc_nz"], z=s["z_nz"], text=s["loc_nz"], customdata=s["cd_nz"],
    hovertemplate=hovertemplate, colorscale=HEAT_SCALE,
    zmin=0, zmax=caps[initial_metric],
    colorbar=dict(title=map_labels[initial_metric]),
    marker_line_color="white", marker_line_width=0.5
))

TITLE_PREFIX = "EU Network for Evidence-Synthesis in the Agrifood Sector: Members by Country"

buttons = []
for m in metrics:
    st = STATE[m]
    buttons.append(dict(
        label=f"{map_labels[m]} ({metric_totals.get(m, 0):,})",
        method="update",
        args=[
            {
                "locations": [st["loc_z"], st["loc_nz"]],
                "z":         [st["z_z"],   st["z_nz"]],
                "text":      [st["loc_z"], st["loc_nz"]],
                "customdata":[st["cd_z"],  st["cd_nz"]],
                "zmin":      [None, 0],
                "zmax":      [None, caps[m]],
                "colorbar":  [None, {"title": {"text": map_labels[m]}}],
            },
            {"title": f"{TITLE_PREFIX} — colouring by {map_labels[m]}"},
        ]
    ))

fig.update_layout(
    title=dict(text=f"{TITLE_PREFIX} — colouring by {map_labels[initial_metric]}",
               x=0.5, xanchor="center"),
    margin=dict(l=0, r=0, t=80, b=0),
    updatemenus=[dict(type="dropdown", x=0.99, xanchor="right",
                      y=1.12, yanchor="top", showactive=True, buttons=buttons)],
    geo=dict(
        projection_type="natural earth",
        showland=True,  landcolor=LAND_GRAY,
        showcountries=True, countrycolor="white",
        showocean=True, oceancolor="white",
        bgcolor="white",
    )
)

fig.show()


# ---------- save interactive HTML ----------
interactive_path = DOCS_DIR / "interactive_country_map.html"
fig.write_html(interactive_path, include_plotlyjs=True, auto_open=False)
print("Saved →", interactive_path.relative_to(ROOT))


# ---------- accessible page (summary + tables) ----------
plot_div = pio.to_html(fig, include_plotlyjs=True, full_html=False)

countries_included   = int(agg_map["gj_name"].nunique())
total_people_all     = int(df.shape[0])
total_people_mapped  = int(agg_map["total_people"].sum())
total_people_unmapped= total_people_all - total_people_mapped
avg_per_country      = round(total_people_mapped / max(1, countries_included), 2)

# ITC stats for summary
if have_itc:
    itc_people_all = int((df["itc_countries"] == "Yes").sum())
    itc_countries_rep = int(agg_map["ITC_flag"].sum())
    itc_denominator = len(ITC_LIST) if ITC_LIST else None
else:
    itc_people_all = 0
    itc_countries_rep = 0
    itc_denominator = None

top5 = (agg_map[["gj_name","total_people"]]
        .sort_values("total_people", ascending=False)
        .head(5).to_records(index=False))
top5_lines = "".join(f"<li>{name}: {count}</li>" for name, count in top5)

mapped_cols = ["gj_name", "total_people", "MC", "Core", "Any_WG"] + wg_cols + ["ITC_flag"]
mapped_table = (agg_map[mapped_cols]
                .rename(columns=table_labels)
                .sort_values("Total", ascending=False))
mapped_html = mapped_table.to_html(index=False, border=0, classes="datatable", table_id="mapped_table")

# unmapped table with all metrics
unmapped_df = df[df["gj_name"].isna()].copy()
if not unmapped_df.empty:
    def entity_label(x): return re.sub(r"\s*\([^)]*\)\s*$", "", str(x or "").strip()).replace("’", "'")
    unmapped_df["Entity"] = unmapped_df[country_col].map(entity_label)

    parts = {
        "Entity": ("Entity", "first"),
        "Total": (country_col, "size"),
        "MC": (mc_col, "sum"),
        "Core": (core_col, "sum"),
        "Any WG": ("any_wg", "sum"),
    }
    wg_table_cols = []
    for c in wg_cols:
        m = re.search(r"\d+", c)
        if not m:
            continue
        short = f"wg{m.group(0)}"
        parts[short] = (c, "sum")
        wg_table_cols.append(short)

    unmapped_agg = unmapped_df.groupby("Entity", as_index=False).agg(**parts).fillna(0)
    for col in ["Total", "MC", "Core", "Any WG"] + wg_table_cols:
        unmapped_agg[col] = unmapped_agg[col].astype(int)

    other_html = (
        "<h2>Not on map (no specific country affiliation)</h2>"
        + unmapped_agg.sort_values("Total", ascending=False)
                      .to_html(index=False, border=0, classes="datatable", table_id="unmapped_table")
    )
else:
    other_html = ""

# Build optional ITC bullets
if have_itc:
    if itc_denominator:
        itc_line1 = f"<li><strong>ITC countries represented:</strong> {itc_countries_rep} / {itc_denominator}</li>"
    else:
        itc_line1 = f"<li><strong>ITC countries represented:</strong> {itc_countries_rep}</li>"
    itc_line2 = f"<li><strong>Participants from ITC countries:</strong> {itc_people_all}</li>"
else:
    itc_line1 = ""
    itc_line2 = ""

page_html = f"""<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8" />
  <title>Members by Country — Interactive Map</title>
  <meta name="viewport" content="width=device-width, initial-scale=1" />
  <style>
    :root {{ --accent: #1F5FCC; }}
    body {{ font-family: system-ui, -apple-system, Segoe UI, Roboto, Arial, sans-serif; margin: 1.25rem; line-height: 1.5; color: #222; }}
    .container {{ max-width: 1200px; margin: 0 auto; }}
    .note {{ color: #333; font-size: 0.95rem; }}
    .datatable {{ border-collapse: collapse; width: 100%; margin-top: 1rem; }}
    .datatable th, .datatable td {{ border-bottom: 1px solid #ddd; padding: 0.5rem; text-align: left; }}
    .datatable tr:hover td {{ background: #f6f8fb; }}
    a:focus, button:focus, [tabindex]:focus {{ outline: 3px solid var(--accent); outline-offset: 2px; }}
    .skip-link {{ position: absolute; left: -9999px; top: auto; width: 1px; height: 1px; overflow: hidden; }}
    .skip-link:focus {{ position: static; width: auto; height: auto; padding: .5rem; background: #fff7cc; border: 1px solid #e0c200; }}
    .sr-only {{ position: absolute; width: 1px; height: 1px; padding: 0; margin: -1px; overflow: hidden; clip: rect(0,0,0,0); border: 0; }}
  </style>
</head>
<body>
  <a class="skip-link" href="#tables">Skip to data tables</a>
  <main class="container">
    <section aria-label="Summary of map metrics">
      <h1>EU Network for Evidence-Synthesis in the Agrifood Sector</h1>
      <p>This page shows an interactive world map of members by country. Hover a country to view totals and Working Group counts.
         Use the dropdown above the map to change the metric (Total, MC, Core, WG1…WG5, ITC).</p>
      <ul>
        <li><strong>Countries included (mapped):</strong> {countries_included}</li>
        <li><strong>Total people (all rows):</strong> {total_people_all}</li>
        <li><strong>On map (countries only):</strong> {total_people_mapped}</li>
        <li><strong>Not on map (no specific country affiliation):</strong> {total_people_unmapped}</li>
        <li><strong>Average per mapped country:</strong> {avg_per_country}</li>
        {itc_line1}
        {itc_line2}
      </ul>
      <details>
        <summary>Top 5 countries by total people</summary>
        <ol>{top5_lines}</ol>
      </details>
      <p class="note">
        Keyboard tip: use Tab to reach the dropdown and plot toolbar (camera icon to download a PNG),
        and arrow keys / +/- to zoom the map. Screen reader users can review the tables below for the same information.
      </p>
    </section>

    <section aria-labelledby="map-heading">
      <h2 id="map-heading">Interactive choropleth map</h2>
      <div role="img"
           aria-label="Choropleth map of members by country. Zero-value countries are shown in light grey; higher values are darker blue. A dropdown lets you switch the metric being mapped.">
        {plot_div}
      </div>
    </section>

    <section id="tables" aria-labelledby="table-heading">
      <h2 id="table-heading">Country counts (mapped)</h2>
      {mapped_html}
      {other_html}
    </section>
  </main>
</body>
</html>"""

accessible_path = DOCS_DIR / "map_accessible.html"
with open(accessible_path, "w", encoding="utf-8") as f:
    f.write(page_html)
print("Saved →", accessible_path.relative_to(ROOT))


Repo root → C:\Users\James\Documents\GitHub\evidence-map-agrifood
Using CSV  → data\processed\data_clean.csv
Using GEO  → data\geo\countries.geojson.json
Saved → outputs\name_match_review.csv
Saved → docs\interactive_country_map.html
Saved → docs\map_accessible.html
